# SmartPay AP – Intelligent Invoice Matching Engine

## AI Architect Case Study

### Author
Harish Kumar Govindu

---

## Objective

This notebook demonstrates a lightweight invoice-to-PO reconciliation engine for the SmartPay AP platform.

The goal is to:
- load invoice and purchase order datasets
- perform intelligent invoice matching
- detect mismatches
- generate explainable reconciliation outcomes
- evaluate matching performance using precision and recall metrics

---

## Business Context

Acme Manufacturing processes approximately 1 million invoices per month across 25 countries.

The current reconciliation process is:
- manual
- time-consuming
- error-prone
- difficult to audit at scale

This notebook demonstrates how lightweight ML-assisted reconciliation can improve:
- operational efficiency
- explainability
- auditability
- AP automation workflows

---

## Design Approach

The reconciliation engine combines:
- deterministic validation rules
- fuzzy vendor similarity matching
- amount consistency checks
- currency validation

The implementation prioritizes:
- explainability
- simplicity
- governance
- enterprise MVP feasibility

# Step 1 — Import Required Libraries

The implementation uses:
- pandas for data processing
- difflib for vendor similarity matching
- scikit-learn metrics for evaluation

The objective is to keep the MVP lightweight, explainable, and easy to operationalize.

In [2]:
import pandas as pd
from difflib import SequenceMatcher
from sklearn.metrics import classification_report

# Step 2 — Load Invoice and PO Datasets

The reconciliation workflow requires:
- invoice records
- purchase order (PO) records

These datasets simulate enterprise AP reconciliation scenarios.

Input sources could originate from:
- PDFs
- Emails
- EDI feeds
- ERP systems

For this MVP implementation, synthetic CSV datasets are used.

In [4]:
# Load datasets
invoices = pd.read_csv("../data/raw/invoices.csv")
po_data = pd.read_csv("../data/raw/po_data.csv")

print(invoices.head())
print(po_data.head())

  invoice_id           vendor  amount currency po_number   tax        date
0   INV-1001         ABC Corp   12000      USD   PO-2201  1200  2026-05-10
1   INV-1002          XYZ Ltd    8500      USD   PO-2202   850  2026-05-11
2   INV-1003     Global Parts   15000      EUR   PO-2203  1500  2026-05-12
3   INV-1004  ABC Corporation   12100      USD   PO-2201  1200  2026-05-13
4   INV-1005  Fast Components    9900      USD   PO-2205   900  2026-05-14
  po_number           vendor  amount currency   tax        date
0   PO-2201         ABC Corp   12000      USD  1200  2026-05-09
1   PO-2202          XYZ Ltd    8500      USD   850  2026-05-10
2   PO-2203     Global Parts   14800      EUR  1500  2026-05-10
3   PO-2205  Fast Components   10000      USD   900  2026-05-13


# Step 3 — Define Matching Strategy

The reconciliation engine uses a hybrid matching approach.

## Rule-Based Validation

The following deterministic validations are applied:
- PO number validation
- currency consistency
- amount threshold validation

## Similarity-Based Matching

Vendor names may vary across invoices and ERP systems.

Example:
- "ABC Corp"
- "ABC Corporation"

To support approximate matching, fuzzy similarity scoring is used.

---

## Why Lightweight ML Instead of Deep Learning?

The MVP prioritizes:
- explainability
- faster implementation
- lower operational complexity
- easier governance and debugging

This aligns with enterprise finance requirements where transparency is critical.

In [5]:
results = []

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# Step 4 — Execute Reconciliation Workflow

For each invoice:
1. Match the corresponding PO record
2. Compare vendor similarity
3. Validate invoice amount
4. Validate currency consistency
5. Generate reconciliation status
6. Produce explainable mismatch reasons

The workflow produces:
- Match status
- Mismatch explanation
- Reconciliation reasoning

In [6]:
for _, invoice in invoices.iterrows():

    po_match = po_data[po_data["po_number"] == invoice["po_number"]]

    # PO not found
    if len(po_match) == 0:
        results.append({
            "invoice_id": invoice["invoice_id"],
            "status": "Mismatch",
            "reason": "PO not found"
        })
        continue

    po = po_match.iloc[0]

    # Vendor similarity
    vendor_score = similarity(invoice["vendor"], po["vendor"])

    # Amount comparison
    amount_difference = abs(invoice["amount"] - po["amount"])

    status = "Matched"
    reasons = []

    # Vendor validation
    if vendor_score < 0.8:
        status = "Mismatch"
        reasons.append("Vendor mismatch")

    # Amount validation
    if amount_difference > 100:
        status = "Mismatch"
        reasons.append("Amount mismatch")

    # Currency validation
    if invoice["currency"] != po["currency"]:
        status = "Mismatch"
        reasons.append("Currency mismatch")

    # Final reconciliation result
    results.append({
        "invoice_id": invoice["invoice_id"],
        "status": status,
        "reason": ", ".join(reasons) if reasons else "Valid Match"
    })

# Evaluation Metrics

The model is evaluated using:
- precision
- recall
- F1-score

These metrics help validate reconciliation quality.

In [8]:
results_df = pd.DataFrame(results)

print(results_df)

  invoice_id    status           reason
0   INV-1001   Matched      Valid Match
1   INV-1002   Matched      Valid Match
2   INV-1003  Mismatch  Amount mismatch
3   INV-1004  Mismatch  Vendor mismatch
4   INV-1005   Matched      Valid Match


In [9]:
y_true = [1,1,1,0,1]
y_pred = [1 if s=="Matched" else 0 for s in results_df["status"]]

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       1.00      0.75      0.86         4

    accuracy                           0.80         5
   macro avg       0.75      0.88      0.76         5
weighted avg       0.90      0.80      0.82         5



# Evaluation Metrics

The invoice reconciliation engine is evaluated using:

- Precision
- Recall
- F1-score

These metrics help measure:
- reconciliation accuracy
- mismatch detection quality
- operational reliability

---

## Why Precision Matters in Financial Systems

False approvals in Accounts Payable systems can lead to:
- payment errors
- compliance violations
- audit risks
- financial leakage

Therefore, precision and explainability are critical for enterprise AP automation platforms.

In [10]:
# Sample evaluation labels
y_true = [1,1,1,0,1]

# Convert reconciliation results into binary predictions
y_pred = [1 if s=="Matched" else 0 for s in results_df["status"]]

# Print evaluation metrics
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       1.00      0.75      0.86         4

    accuracy                           0.80         5
   macro avg       0.75      0.88      0.76         5
weighted avg       0.90      0.80      0.82         5

